In [1]:
# import packages
import os
from cellpose import models
from cellpose import denoise, io
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["JAVA_HOME"] ="/home/manjari/Downloads/Fiji.app/java/linux-amd64/zulu8.60.0.21-ca-fx-jdk8.0.322-linux_x64/jre/lib/amd64/server/"
# import packages
import os
import pickle
#from starmap.config import Config
#from starmap.pipeline import Pipeline
import starmap.io as io
import h5py
import matplotlib.pyplot as plt
import numpy as np
import json
import math
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import napari
# import IPython.display
import bardensr
import bardensr.plotting
import bardensr.preprocessing
# import imagej
import copy
from datetime import datetime
import shutil
import imagej
from cellpose import plot
from cellpose import utils, io
import tifffile
import argparse
import warnings
warnings.simplefilter('ignore', pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
import starmap.io as io
import starmap.preprocessing as preproc
import starmap.gene_calling as genecall
import starmap.cell_segmentation as cellseg
from starmap.config import Config
import IPython.display
import PIL
from PIL import Image, ImageDraw
import skimage as sk
import cv2

### Function

In [2]:
def filter_cell_size(barcode_masks, min_area = 1000):
    filtered_masks = []
    for i in range(len(barcode_masks)):
# Ensure barcode_mask is a NumPy array
        barcode_masks[i] = np.array(barcode_masks[i])

        # Label the segmented cells
        labeled_masks = sk.measure.label(barcode_masks[i])

        # Measure region properties
        regions = sk.measure.regionprops(labeled_masks)  #original is regionprops  
        # Create a list to store the filtered cells
        filtered_cells = []

        for region in regions:
            if region.area >= min_area:
                filtered_cells.append(region)

        # Create a new mask with only the large cells
        filtered_mask = np.zeros_like(barcode_masks[i])

        for region in filtered_cells:
            filtered_mask[labeled_masks == region.label] = region.label

        filtered_masks.append(filtered_mask)
    return filtered_masks
from skimage.morphology import opening, disk, remove_small_objects
from skimage.measure import label, regionprops
import numpy as np

def filter_cell_size_modify(barcode_masks, min_area=1000, open_radius=2):
    filtered_masks = []
    for i in range(len(barcode_masks)):
        m = np.asarray(barcode_masks[i]) > 0          # ensure binary boolean

        # 1) break thin 1-pixel bridges so dots detach from big objects
        if open_radius > 0:
            m = opening(m, disk(open_radius))

        # 2) remove small connected components directly (simpler + reliable)
        m = remove_small_objects(m, min_size=min_area)

        # 3) relabel (use connectivity=1 to avoid diagonal merging)
        labeled_masks = label(m, connectivity=1)

        filtered_masks.append(labeled_masks.astype(np.int32))
    return filtered_masks
#denoise images
def segment_barcode_images(dapi_merged, diameter=28, channels=[0,0], threshold_low = 200, threshold_high = 1200): #0-255 intensity values # original (600,2000)
    dapi_merged = [np.clip(np.sum(dapi_merged[i][0], axis = 0), threshold_low, threshold_high) for i in range(len(dapi_merged))]

    model = models.Cellpose(model_type='cyto3') 
    masks, flows, styles, dapi_merged_dn = model.eval(dapi_merged, diameter=diameter, channels=channels)
    return masks

def open_hdf5(file_path): #assumes this is a list of np arrays
    var_name= []
    f = h5py.File(file_path)
    print(f)
    data = f['list']
    for i in range(len(data)):
        print(i)
        var_name.append(data[str(i)][:])
    f.close()
    return var_name

def save_hdf5(matrix, file_path): #assumes this is a list of np arrays
    f = h5py.File(file_path,'w')
    grp = f.create_group('list')
    for i,arr in enumerate(matrix):
        grp.create_dataset(str(i), data=arr)
    f.close()
    return None

def colorbleed_correction_inversemat(rounds_aligned, colormixing_matrix=np.array([
    [1,.657,0,0],  # matrix[0,1] is positive because whenever frame 0 is bright, frame 1 is also a bit bright
    [0.38,1,0.28,0.28],
    [0,0.01,1,.07],
    [0,0.08,.4,1]])):
    fix=np.linalg.inv(colormixing_matrix)

    for i in range(len(rounds_aligned)):
        print("FOV: "+str(i))
        rounds_aligned[i] = np.clip(np.einsum('rcxy,cd->rdxy',rounds_aligned[i],fix),0,None)

    return rounds_aligned
#TO DO: apply image masks to dapi mask segmentation
def apply_mask_to_images_dapi(image_array, mask):
    mask_cropped = []
    for i in range(len(image_array)):
        mask_width, mask_height = mask[i].shape
        x_index, y_index = np.where(mask[i] == 1)
        x_st, x_ed, y_st, y_ed = min(x_index), max(x_index), min(y_index), max(y_index)
        result_array = np.zeros((mask_width, mask_height), dtype=image_array[i].dtype)

        resized_image = np.zeros((mask_width, mask_height), dtype=image_array[i].dtype)
        resized_image = image_array[i]*mask[i]
        mask_cropped.append(resized_image)

    return mask_cropped

def apply_mask_to_images(image_array, mask):
    mask_width, mask_height = mask.shape
    x_index, y_index = np.where(mask == 1)
    x_st, x_ed, y_st, y_ed = min(x_index), max(x_index), min(y_index), max(y_index)
    result_array = np.zeros((image_array.shape[0], image_array.shape[1], mask_width, mask_height), dtype=image_array.dtype)

    for i in range(image_array.shape[0]):
        for j in range(image_array.shape[1]):
            x_core, y_core = image_array[i, j].shape[0], image_array[i, j].shape[1]
            if (x_ed-x_st+1) != x_core or (y_ed-y_st+1) != y_core:
                print(datetime.now().strftime("%d/%m/%Y %H:%M:%S"), "        "+"masks not fit!")
            resized_image = np.zeros((mask_width, mask_height), dtype=image_array.dtype)
            resized_image[x_st:x_ed+1, y_st:y_ed+1] = image_array[i, j]
            result_array[i, j] = resized_image
            
    return result_array

## Finding cell barcodes - manjari

In [3]:
#ALL NECESSARY FUNCTIONS FOR ASSIGNING CELL TO BARCODE


import pandas as pd
import numpy as np

def create_barcoded_cells_df(barcoded_cells, most_overlapping_cells, fov_round_intensity_bin_np, binary_to_string):
    # Initialize the DataFrame
    barcoded_cells_df = pd.DataFrame(columns = ['FOV', 'barcode_cell', 'dapi_cell', 'barcode'])
    barcoded_cells_df['FOV'] = np.zeros(len(barcoded_cells))
    barcoded_cells_df['barcode_cell'] = np.zeros(len(barcoded_cells))
    barcoded_cells_df['dapi_cell'] = np.zeros(len(barcoded_cells))
    barcoded_cells_df['barcode'] = np.zeros(len(barcoded_cells))

    # Loop through each barcode cell in the list
    for i in range(len(barcoded_cells)):
        # Extract the FOV and barcode cell index from barcoded_cells
        fov_index = barcoded_cells[i][0]
        barcode_cell_index = barcoded_cells[i][1]

        # Set FOV and barcode_cell in the DataFrame
        barcoded_cells_df['FOV'][i] = fov_index
        barcoded_cells_df['barcode_cell'][i] = barcode_cell_index

        # Find the dapi cell that overlaps with the current barcode cell
        # Get the list of overlapping cells for the current FOV
        overlapping_cells_for_fov = most_overlapping_cells[fov_index]

        # Search for the dapi cell that corresponds to the current barcode_cell_index
        for barcode_cell_id, dapi_cell_id, iou in overlapping_cells_for_fov:
            if barcode_cell_id == barcode_cell_index:
                # Assign the corresponding dapi_cell_id to the DataFrame
                barcoded_cells_df['dapi_cell'][i] = dapi_cell_id
                break  # Exit once the matching barcode cell is found

        # Convert binary to string (assuming fov_round_intensity_bin_np is predefined)
        base_barcode = binary_to_string(fov_round_intensity_bin_np[fov_index][barcode_cell_index])

        # Assign the barcode string to the DataFrame
        barcoded_cells_df['barcode'][i] = base_barcode

    # Filter out rows where dapi_cell is less than or equal to 0
    barcoded_cells_df_filtered = barcoded_cells_df[barcoded_cells_df['dapi_cell'] > 0]
    
    return barcoded_cells_df_filtered

# Example usage:
# dapi_mask_cropped_list and barcode_mask_cropped_list should be lists of 2D numpy arrays of masks
# Example:
# all_most_overlapping_cells = find_most_overlapping_cells(dapi_mask_cropped_list, barcode_mask_cropped_list)

def find_second_largest_intensity_index(intensity):
    first_largest=second_largest = float('-inf')
    first_index=second_index=-1
    for i, num in enumerate(intensity):
        if num > first_largest:
            second_largest = first_largest
            second_index = first_index
            first_largest = num
            first = i
        elif num > second_largest and num != first_largest:
            second_largest = num
            second_index = i
    return second_largest, second_index 

def third_largest_intensity_index(intensity):
    if len(intensity) < 3:  # Check if there are at least 3 elements
        return None, None
    
    # Use a set to get unique values and then sort them
    unique_values = sorted(set(intensity), reverse=True)
    
    if len(unique_values) < 3:  # Check if there are at least 3 distinct numbers
        return None, None
    
    third_largest = unique_values[2]  # Get the third largest
    third_largest_index = np.where(intensity == third_largest)[0]  # Get index of third largest

    return third_largest, third_largest_index.tolist()

def fourth_largest_intensity_index(intensity):
    if len(intensity) == 0:  # Check if the array is empty
        return None, None
    
    # Find the smallest number
    smallest = np.min(intensity)  # or use min(arr) for a list
    
    # Get the index of the smallest number
    smallest_index = np.where(intensity == smallest)[0]  # Find all indices
    
    return smallest, smallest_index.tolist()

#finds potential barcoded cells across all FOVs
def potential_cells(all_cells_channel_intensity, binary_intensity, threshold_difference): 
    barcoded_cells = []
    barcoded_cells_int = []
    barcoded_cells_bin = []

    for fov in range(len(all_cells_channel_intensity)):
        for cell in range(len(all_cells_channel_intensity[fov])):
            rounds_met_condition=0
            for round in range(len(all_cells_channel_intensity[fov][cell])):
                condition_met_in_round = False  # To track if any condition is met in this round
                if (abs(all_cells_channel_intensity[fov][cell][round][0] - all_cells_channel_intensity[fov][cell][round][1]) > threshold_difference or
                    abs(all_cells_channel_intensity[fov][cell][round][0] - all_cells_channel_intensity[fov][cell][round][2]) > threshold_difference or
                    abs(all_cells_channel_intensity[fov][cell][round][0] - all_cells_channel_intensity[fov][cell][round][3]) > threshold_difference or
                    abs(all_cells_channel_intensity[fov][cell][round][1] - all_cells_channel_intensity[fov][cell][round][2]) > threshold_difference or
                    abs(all_cells_channel_intensity[fov][cell][round][1] - all_cells_channel_intensity[fov][cell][round][3]) > threshold_difference or
                    abs(all_cells_channel_intensity[fov][cell][round][2] - all_cells_channel_intensity[fov][cell][round][3]) > threshold_difference):
                    condition_met_in_round = True
                # If any condition is met in this round, increase the count
                if condition_met_in_round:
                    rounds_met_condition += 1   

             # If the condition is met in all rounds, add this cell to the list
            if rounds_met_condition == len(all_cells_channel_intensity[fov][cell]):
                barcoded_cells.append([fov, cell])
                barcoded_cells_bin.append(binary_intensity[fov][cell])
                barcoded_cells_int.append(all_cells_channel_intensity[fov][cell])
    #barcoded_cells = np.unique(barcoded_cells, axis = 0)
    #barcoded_cells_int = np.unique(barcoded_cells_int, axis = 0)
    #barcoded_cells_bin = np.unique(barcoded_cells_bin, axis = 0)
    print(len(barcoded_cells_bin))
    return barcoded_cells, barcoded_cells_int, barcoded_cells_bin


def assign_highest_to_one(arr):    
    max_value = max(arr)  # Find the highest value
    return [1 if x == max_value else 0 for x in arr]

def find_cell_barcodes(barcode_image, masks, num_rounds):
    fov_round_intensity = []
    fov_round_intensity_bin = []
    for fov in range(len(masks)):
        single_round_intensity = []
        single_round_intensity_bin = []
        for cell in np.unique(masks[fov]):

            round_intensity = []
            round_intensity_bin = []
            for round in range(num_rounds):
                cell_location = np.where(masks[fov] == cell)
                #print("cell location:" + str(cell_location))
                #print(barcode_image[fov][round][:,cell_location[0], cell_location[1]])
                channel_info = np.mean(barcode_image[fov][round][:,cell_location[0], cell_location[1]], axis = 1)
                round_intensity.append(channel_info) #single cell all round intensities
                round_intensity_bin.append(assign_highest_to_one(channel_info))
            single_round_intensity.append(np.array(round_intensity)) # all cells in an fov, all round intensities
            single_round_intensity_bin.append(np.array(round_intensity_bin))
        fov_round_intensity.append(np.array(single_round_intensity)) # all_fovs, all cells, all round intensities
        fov_round_intensity_bin.append(np.array(single_round_intensity_bin))
    #fov_round_intensity_np = np.stack(fov_round_intensity)
    #fov_round_intensity_bin_np = np.stack(fov_round_intensity_bin)

    return fov_round_intensity, fov_round_intensity_bin


def binary_to_string(matrix):
    # Define the mapping
    mapping = {
        (0, 1, 0, 0): 'T',
        (1, 0, 0, 0): 'G',
        (0, 0, 1, 0): 'C',
        (0, 0, 0, 1): 'A'
    }
    
    # Initialize an empty string to hold the result
    result = ""
    
    for row in matrix:
        # Convert the list to a tuple for dictionary lookup
        key = tuple(row)
        # Append the corresponding character or a placeholder if not found
        result += mapping.get(key, '?')  # '?' for unmapped rows

    return result

import numpy as np
from skimage.measure import label

def renumber_labels(mask):
    """
    Renumber the labels in the mask so that they form a consecutive sequence starting from 1.
    
    Args:
    - mask (numpy array): The input mask with labeled components.
    
    Returns:
    - new_mask (numpy array): The mask with consecutive labels.
    """
    # Label the mask and get the unique labels
    original_labels = np.unique(mask)
    
    # If the background is labeled as 0, we need to exclude it
    original_labels = original_labels[original_labels > 0]
    
    # Create a mapping from the original labels to consecutive labels
    label_mapping = {original_label: new_label for new_label, original_label in enumerate(original_labels, start=1)}
    
    # Create the new mask by applying the mapping to the original mask
    new_mask = np.copy(mask)
    for original_label, new_label in label_mapping.items():
        new_mask[mask == original_label] = new_label
    
    return new_mask

# Example usage:
# Assuming barcode_mask_cropped is a list of 2D numpy arrays (one for each image/field of view)
import numpy as np
from scipy.ndimage import label
from skimage.measure import regionprops

def find_most_overlapping_cells(dapi_mask_cropped_list, barcode_mask_cropped_list):
    # Function to calculate Intersection over Union (IoU)
    def calculate_iou(mask1, mask2):
        intersection = np.sum(mask1 & mask2)  # Pixel overlap
        union = np.sum(mask1 | mask2)  # Total area covered by either mask
        return intersection / union if union != 0 else 0
    
    # List to store the most overlapping dapi cell index for each barcode cell across all images
    all_most_overlapping_cells = []

    # Iterate over each pair of dapi and barcode masks (for each FoV/image)
    for dapi_mask_cropped, barcode_mask_cropped in zip(dapi_mask_cropped_list, barcode_mask_cropped_list):
        
        # List to store the most overlapping dapi cell index for each barcode cell in this FoV
        most_overlapping_cells = []

        # Iterate over each barcode cell in the current FoV (skip background label 0)
        for barcode_cell_id in np.unique(barcode_mask_cropped):
            if barcode_cell_id == 0:
                continue  # Skip background cells (label 0)

            barcode_mask = (barcode_mask_cropped == barcode_cell_id)  # Binary mask for the current barcode cell

            # Initialize the best overlap (most IoU) and corresponding dapi cell id
            best_overlap = 0
            best_dapi_cell_id = -1

            # Find the DAPI cells that share pixels with the barcode cell
            overlap_pixels = np.where(barcode_mask)  # Get coordinates of pixels where barcode cell is present
            dapi_cells_in_overlap = np.unique(dapi_mask_cropped[overlap_pixels])  # Find DAPI cells in the overlapping pixels

            # Remove background label (0) from the list of overlapping DAPI cells
            dapi_cells_in_overlap = dapi_cells_in_overlap[dapi_cells_in_overlap > 0]

            # Iterate over only the DAPI cells that overlap with the current barcode cell
            for dapi_cell_id in dapi_cells_in_overlap:
                dapi_mask = (dapi_mask_cropped == dapi_cell_id)  # Binary mask for the current dapi cell

                # Calculate the IoU for the overlap between the current barcode and dapi cell
                iou = calculate_iou(barcode_mask, dapi_mask)

                # If this overlap is the largest found so far, update the best overlap and cell
                if iou > best_overlap:
                    best_overlap = iou
                    best_dapi_cell_id = dapi_cell_id

            # Append the best matching DAPI cell for the current barcode cell in this FoV
            most_overlapping_cells.append((barcode_cell_id, best_dapi_cell_id, best_overlap))

        # Append the most overlapping cells for this particular FoV/image
        all_most_overlapping_cells.append(most_overlapping_cells)

    return all_most_overlapping_cells


### Filter cell by size

In [ ]:
def filter_cell_size1(barcode_masks, min_area=400):
    filtered_masks = []

    for i in range(len(barcode_masks)):
        barcode_masks[i] = np.array(barcode_masks[i])

        
        labeled_masks = sk.measure.label(barcode_masks[i])

      
        regions = sk.measure.regionprops(labeled_masks)

       
        label_mapping = {}

       
        for region in regions:
            if region.area >= min_area:
             
                original_label = np.unique(barcode_masks[i][labeled_masks == region.label])
                original_label = original_label[original_label > 0]  

                if len(original_label) > 0:
                    label_mapping[region.label] = original_label[0]  

       
        filtered_mask = np.zeros_like(barcode_masks[i])

        for region in regions:
            if region.label in label_mapping:
                filtered_mask[labeled_masks == region.label] = label_mapping[region.label]

        filtered_masks.append(filtered_mask)

    return filtered_masks



In [7]:
### NEW Function for Round Alignment with OpenCV
def round_alignment_opencv(rounds, round_align=0): # assumes structure is fov x round x channel x X x Y
    # (AFTER BACKGROUND SUBTRACTION)
    # Detect keypoints and compute descriptors
    #rounds = np.stack(rounds, axis = 0)[:, round_index] #added

    images = rounds#copy.deepcopy((rounds/np.max(rounds))*255).astype(np.uint8)
    params = cv2.SimpleBlobDetector_Params()

    # Change thresholds
    params.blobColor = 255 # lighter blobs
    params.minThreshold = 0
    params.maxThreshold = 255

    # Filter by Area.
    params.filterByArea = True
    params.maxArea = 1000000

    # Filter by Circularity
    params.filterByCircularity = True
    params.minCircularity = 0.5

    # Filter by Inertia
    params.filterByInertia = True
    params.minInertiaRatio = 0.01

    detector = cv2.SimpleBlobDetector_create(params)
    transforms = []
    #images = (np.array(images > 0)*255).astype(np.uint8)
    for i in range(len(images)-1):
        r1 = images[0]
        rn = images[i+1]
        r1 = (((r1-np.min(r1))/(np.max(r1)-np.min(r1)))*255).astype(np.uint8)
        rn = (((rn-np.min(rn))/(np.max(rn)-np.min(rn)))*255).astype(np.uint8)
        print(r1.dtype)
        #rn = np.clip(rn, 10, 200)#np.percentile(rn.ravel(), 95))

        #r1 = np.clip(r1, 10, 200)#np.percentile(r1.ravel(), 95))

        sift = cv2.SIFT_create()
        # find the keypoints and descriptors with SIFT
        kp1, des1 = sift.detectAndCompute(rn,None)
        kp2, des2 = sift.detectAndCompute(r1,None)
        # FLANN parameters
        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm = FLANN_INDEX_KDTREE, trees = 5)
        search_params = dict(checks=50) # or pass empty dictionary
        flann = cv2.FlannBasedMatcher(index_params,search_params)
        matches = flann.knnMatch(des1,des2,k=2)
        # Need to draw only good matches, so create a mask
        #matchesMask = [[0,0] for i in range(len(matches))]
        # ratio test as per Lowe's paper
        good = []
        for p,(m,n) in enumerate(matches):
            if m.distance < 0.8*n.distance:
                good.append(m)
        # Sort matches by score
        #matches = sorted(matches, key=lambda x: x.distance)
        print('Number of matches between round 1 and ' + str(i+1)+": "+ str(len(good)))
        #img3 = cv2.drawMatches(r11,kp1,rn1,kp2,good[:5],None,flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        #plt.imshow(img3),plt.show()
        # Extract matched keypoints
        src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
        # Find homography
        H, _ = cv2.estimateAffinePartial2D(src_pts, dst_pts)
        print(i)
        #print(j)
        print(images[i+1].shape)
        #images[i+1] = cv2.warpAffine(images[i+1], H, (images[i+1].shape[1], images[i+1].shape[0])) #original code
        images[i+1] = cv2.warpAffine(images[i+1], H, (images[i+1].shape[1], images[i+1].shape[0]))
    
    return images, H

def apply_transform(image, H):
    image = cv2.warpAffine(image, H, (image.shape[1], image.shape[0]))
    return image

In [ ]:
import numpy as np
from cellpose import models

def segment_barcode_images1(dapi_merged, diameter=28, channels=[0,0], 
                           threshold_low=0, threshold_high=1500, flow_threshold  = 0.4):
    
    # 1. Save dapi_merged to numpy array
    dapi_merged = np.array(dapi_merged)
    
    # 2. Combine all channels in every rounds, and sum 4 channels
    #   get the dimension (region, round, height, width)
    combined_rounds = np.sum(dapi_merged, axis=2)
    
    # 3. Combine two rounds max signal
   
    composite_images = np.maximum(combined_rounds[:, 0, :, :],
                                  combined_rounds[:, 1, :, :])
    
    # 4. Crop image using the value between threshold_low and threshold_high 
    composite_images = np.clip(composite_images, threshold_low, np.percentile(composite_images, 99))#threshold_high)
    
    
    
    # 5.using Cellpose to do cell segmentation
    model = models.Cellpose(model_type='cyto3')

    mask_list = []
    for region_img in composite_images:
        # Segment cell by region
        masks, flows, styles, img_dn = model.eval(region_img, diameter=diameter, channels=channels, flow_threshold  = flow_threshold)
        
      
        if masks.ndim == 2:
            binary_mask = (masks > 0).astype(np.uint8)
        else:
       
            binary_mask = (np.sum(masks, axis=0) > 0).astype(np.uint8)
        
        mask_list.append(binary_mask)
    
   
    region_masks = np.stack(mask_list, axis=0)
    return region_masks

In [ ]:
import numpy as np
from cellpose import models

def segment_barcode_images2(dapi_merged, diameter=25, channels=[1,3], 
                           threshold_low=0, threshold_high=1200,
                           flow_threshold=0.4, cellprob_threshold=-2):
  
   
    # 1. Save dapi_merged to numpy array
    dapi_merged = np.array(dapi_merged)

    # 2. Combine all channels in every round, and sum 4 channels
    #   get the dimension (region, round, height, width)
    combined_rounds = np.sum(dapi_merged, axis=2)

    # 3. Combine two rounds max signal
    composite_images = np.maximum(combined_rounds[:, 0, :, :],
                                  combined_rounds[:, 1, :, :])

    # 4. Clip image using the value between threshold_low and threshold_high
    composite_images = np.clip(composite_images, threshold_low, threshold_high)

    
    composite_images = (composite_images - threshold_low) / (threshold_high - threshold_low)
    composite_images = (composite_images * 255).astype(np.uint8)

    # 5. Using Cellpose to do cell segmentation
    model = models.Cellpose(model_type='cyto2')

    mask_list = []
    for region_img in composite_images:
        # Segment cell by region
        masks, flows, styles, img_dn = model.eval(region_img, diameter=diameter, channels=channels,
                                                  flow_threshold=flow_threshold,
                                                  cellprob_threshold=cellprob_threshold)
        
        
        if masks.ndim == 2:
            binary_mask = (masks > 0).astype(np.uint8)
        else:
            binary_mask = (np.sum(masks, axis=0) > 0).astype(np.uint8)
        
        mask_list.append(binary_mask)

    # 6. Stack each region’s mask, retaining the region dimension
    region_masks = np.stack(mask_list, axis=0)
    return region_masks

## Read in Masks of DAPI image and Barcode Images

In [10]:
# import packages
import os
from cellpose import models
from cellpose import denoise, io
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["JAVA_HOME"] ="/home/manjari/Downloads/Fiji.app/java/linux-amd64/zulu8.60.0.21-ca-fx-jdk8.0.322-linux_x64/jre/lib/amd64/server/"
# import packages
import os
import pickle
#from starmap.config import Config
#from starmap.pipeline import Pipeline
import starmap.io as io
import h5py
import matplotlib.pyplot as plt
import numpy as np
import json
import math
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import napari
# import IPython.display
import bardensr
import bardensr.plotting
import bardensr.preprocessing
# import imagej
import copy
from datetime import datetime
import shutil
import imagej
from cellpose import plot
from cellpose import utils, io
import tifffile
import argparse
import warnings
warnings.simplefilter('ignore', pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)
import starmap.io as io
import starmap.preprocessing as preproc
import starmap.gene_calling as genecall
import starmap.cell_segmentation as cellseg
from starmap.config import Config
import IPython.display
import PIL
from PIL import Image, ImageDraw
import skimage as sk
import cv2

In [24]:
import h5py
Brain="Brain1"
barcode_file_path=r"Y:\Huihui\HH\BARseq2_01202026\Analysis_BARseq2_Transcriptome\Barcode_Gene\\"
#Gene cell segmentation masks
gene_masks = open_hdf5(barcode_file_path+ Brain+"\\STARmap_output\\mask_expanded.hdf5")

#Barcode nissl masks
barcode_nissl_masks = open_hdf5(barcode_file_path+ Brain + "_1"+"\\STARmap_output\\mask_expanded.hdf5")


#read in barcode image masks from alignment 
reg_masks = open_hdf5(barcode_file_path + Brain + "_1"+ "\\STARmap_output\\image_masks.hdf5")

#Read in barcode colorbleeding corrected image
barcode_cropped_image = open_hdf5(barcode_file_path + Brain + "_1"+ "\\STARmap_output\\image_corrected.hdf5")

#dapi = np.load(r"W:\Huihui\012525\Mouse1\\" + Brain + "\\STARmap_loadImg\\dapi.npy")

gene_nissl = np.load(barcode_file_path + "\\"+ Brain + "\\STARmap_loadImg\\dapi.npy")
barcode_nissl = np.load(barcode_file_path+ "\\" + Brain + "_1"+ "\\STARmap_loadImg\\dapi.npy")

<HDF5 file "mask_expanded.hdf5" (mode r)>
0
1
2
3
4
<HDF5 file "mask_expanded.hdf5" (mode r)>
0
1
2
3
4
<HDF5 file "image_masks.hdf5" (mode r)>
0
1
2
3
4
<HDF5 file "image_corrected.hdf5" (mode r)>
0
1
2
3
4


### Barcode normalization

In [ ]:
import h5py
import numpy as np
import bardensr.preprocessing  
import starmap.preprocessing as preproc
import starmap.io as io

In [ ]:



dataset_key = 'list/0'         

var_name=[]
for i in range(len(barcode_cropped_image)):
    print(i)
      
    data = barcode_cropped_image[i]
      
    Xflat = data.reshape(-1, data.shape[-2], data.shape[-1])
   
    Xflat = Xflat[:, np.newaxis, :, :]
   
    pixel_max = Xflat[:, 0, :, :].max(axis=0)  
    signal_mask = pixel_max > 600  
    Xflat[:, 0, ~signal_mask] = 0  
    Xnorm = bardensr.preprocessing.minmax(Xflat)

    Xnorm = bardensr.preprocessing.minmax(Xnorm)
   

    Xnorm = Xnorm.reshape(data.shape)

    var_name.append(Xnorm)
#f.close()
barcode_cropped_image_norm=var_name

image_preped = []
for i in range(len(barcode_cropped_image_norm)):
    image_preped.append(preproc.apply_mask_to_images(barcode_cropped_image_norm[i], reg_masks[i]))
io.save_hdf5(image_preped, barcode_file_path + Brain + "_1\\STARmap_output\\image_preped.hdf5")
barcode_image_norm=image_preped

barcode_mask=segment_barcode_images1(barcode_image_norm, diameter = 50, threshold_low= 0.015, threshold_high=10) #originally 5 threshold_low= 0.035


barcode_image_align_transform=[]
aligned_gene_nissl_masks=[]
aligned_barcode_nissl_masks=[]
barcode_cropped_image_Transform=[]
barcode_nissl_masks_transformed=[]
aligned_gene_barcode_nissl=[]
images=barcode_image_norm

aligned_gene_barcode_nissl = list(aligned_gene_barcode_nissl)
for i in range(len(gene_nissl)):
    all_nissl = np.stack([gene_nissl[i], barcode_nissl[i]], axis = 0)
    aligned_masks1, transform =round_alignment_opencv(all_nissl)
    aligned_gene_barcode_nissl.append(aligned_masks1)
    #aligned_gene_nissl_masks.append(aligned_masks1[0])
    #aligned_barcode_nissl_masks.append(aligned_masks1[1])
    barcode_image_align_fov = apply_transform(barcode_mask[i], transform)
    barcode_image_align_transform.append(barcode_image_align_fov )
    barcode_nissl_masks_fov=apply_transform(barcode_nissl_masks[i], transform)
    barcode_nissl_masks_transformed.append(barcode_nissl_masks_fov)
    
    for j in range(len(images[0])):
        images[i][ j, 0] = cv2.warpAffine(images[i][ j, 0], transform, (images[i][ j, 0].shape[1], images[i][ j, 0].shape[0]))
        images[i][ j, 1] = cv2.warpAffine(images[i][ j, 1], transform, (images[i][ j, 1].shape[1], images[i][ j, 1].shape[0]))
        images[i][ j, 2] = cv2.warpAffine(images[i][ j, 2], transform, (images[i][ j, 2].shape[1], images[i][ j, 2].shape[0]))
        images[i][ j, 3] = cv2.warpAffine(images[i][ j, 3], transform, (images[i][ j, 3].shape[1], images[i][ j, 3].shape[0]))  

aligned_gene_barcode_nissl=np.transpose(aligned_gene_barcode_nissl,(1,0,2,3))

filtered_masks = filter_cell_size(barcode_image_align_transform, min_area = 1000)
barcode_mask_consecutive = [renumber_labels(mask) for mask in filtered_masks]

fov_round_intensity_raw_np, fov_round_intensity_bin_np = find_cell_barcodes(images, barcode_mask_consecutive, 2) #round_num 2 original code

barcoded_cells, barcoded_cells_int, barcoded_cells_bin = potential_cells(fov_round_intensity_raw_np,fov_round_intensity_bin_np, 0.015) #threshold 150 0.015

most_overlapping_cells = find_most_overlapping_cells(gene_masks, barcode_mask_consecutive) # original code

#hh

barcoded_cells_filtered = []
fov_round_intensity_bin_np_filtered = []
for [i,j] in barcoded_cells:
    temp_intensities=fov_round_intensity_raw_np[i][j][0]
    #print(temp_intensities)
    max_intensity= max(temp_intensities)
    if max_intensity > 0.0015: #if you have three rounds use (if cal_value > threshold and cal_value1 > threshold and cal_value2 > threshold:)
        barcoded_cells_filtered.append([i,j])

barcoded_cells_df_filtered = create_barcoded_cells_df(barcoded_cells_filtered, most_overlapping_cells, fov_round_intensity_bin_np, binary_to_string)

barcode_csv = pd.read_csv(r"Y:\Huihui\HH\STARmap01202026\Analysis_BARseq3_Transcriptome\Barcode_Gene\Brain1_1\Barcode_List.csv", header = None)
barcode_csv['3rounds'] = [i[:2] for i in np.array(barcode_csv[1])] 
print(barcode_csv['3rounds'])
barcoded_cells_df_filtered['virus'] = barcoded_cells_df_filtered['barcode'].map(
    barcode_csv.set_index('3rounds')[0]
)
print(barcoded_cells_df_filtered['virus'] )

gene_df= pd.read_csv(barcode_file_path + Brain +"\\STARmap_output\\gene_mapped.csv")

barcoded_cells_df_filtered['cell_ID'] = barcoded_cells_df_filtered['dapi_cell']
merged_df = pd.merge(barcoded_cells_df_filtered, gene_df, on=['FOV', 'cell_ID'], how='outer')
merged_df = merged_df.drop(['barcode_cell','dapi_cell', 'barcode'], axis=1)

Fina_list=merged_df[merged_df['virus'].notna()]

Fina_list.to_csv(barcode_file_path + Brain+"_1"+"\\STARmap_output\\Barcodelist.csv")